In [1]:
import json, shutil, os, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from threadpoolctl import threadpool_limits
threadpool_limits(limits=1)

In [2]:
TASK = Path.cwd().parent
INPUT, OUTPUT = TASK / 'input', TASK / 'output'

In [3]:
config = json.loads((INPUT / 'config.json').read_text())
SEED = config['seed']
rng = np.random.default_rng(SEED)

In [4]:
for cell in json.loads((INPUT / 'helpers.ipynb').read_text())['cells']:
    exec(''.join(cell['source']), globals())
carry(INPUT, OUTPUT)

In [5]:
data = np.load(INPUT / 'digits.npz')
x, y, xt, yt = (data[k] for k in ['x_train', 'y_train', 'x_test', 'y_test'])

In [6]:
# gradient checks on each parameter group before any optimization
tiny = initialize(rng, 5, 3)
loss, analytical = loss_grad(tiny, x[:7], y[:7])
gradient_rows = []
for key in tiny:
    index = tuple(0 for _ in tiny[key].shape)
    original = tiny[key][index]
    tiny[key][index] = original + 1e-5
    plus = loss_grad(tiny, x[:7], y[:7])[0]
    tiny[key][index] = original - 1e-5
    minus = loss_grad(tiny, x[:7], y[:7])[0]
    tiny[key][index] = original
    numerical = (plus - minus) / 2e-5
    gradient_rows.append({'parameter': key, 'analytical': analytical[key][index], 'numerical': numerical,
                          'absolute_error': abs(analytical[key][index] - numerical)})
gradient_checks = pd.DataFrame(gradient_rows)
assert gradient_checks.absolute_error.max() < 1e-7
gradient_checks.to_csv(OUTPUT / 'gradient_checks.csv', index=False)

In [7]:
root = initialize(rng, config['width'], config['blocks'])
train(root, x, y, config['root_steps'], rng, config['root_lr'], config['batch'])
nets, rows = {'root': root}, []
rows.append({'model': 'root', 'role': 'common_ancestor', 'source': -1, **digit_metrics(root, xt, yt)})
for source in range(config['all_sources']):
    # distinct minibatch histories and class prevalence; one common digit task
    class_weights = np.exp(0.8 * rng.normal(size=config['classes']))
    probabilities = class_weights[y] / class_weights[y].sum()
    family = clone(root)
    train(family, x, y, config['family_steps'], rng, config['branch_lr'], config['batch'], probabilities)
    name = f'family{source}'
    nets[name] = family
    rows.append({'model': name, 'role': 'family_ancestor', 'source': source, **digit_metrics(family, xt, yt)})
    roles = ['cal_donor', 'test_donor'] + [f'reference{j}' for j in range(config['references_per_source'])]
    for role in roles:
        child = clone(family)
        bootstrap = rng.choice(len(y), len(y), replace=True, p=probabilities)
        steps = config['reference_steps'] if role.startswith('reference') else config['donor_steps']
        train(child, x[bootstrap], y[bootstrap], steps, rng, config['branch_lr'], config['batch'])
        name = f'{role}{source}'
        nets[name] = child
        rows.append({'model': name, 'role': role, 'source': source, **digit_metrics(child, xt, yt)})
save_nets(OUTPUT / 'trained_networks.npz', nets)
pd.DataFrame(rows).to_csv(OUTPUT / 'network_metrics.csv', index=False)

In [8]:
# fixed homologous marker coordinates; shared by every query and reference
dimensions = config['width'] ** 2 + config['width']
marker_order = rng.permutation(dimensions)
np.save(OUTPUT / 'marker_order.npy', marker_order)
pd.DataFrame(rows).groupby('role')[['digit_accuracy', 'digit_nll']].agg(['mean', 'min', 'max'])

digit_accuracy            ... digit_nll          
                          mean       min  ...       min       max
role                                      ...                    
cal_donor             0.971111  0.964444  ...  0.100732  0.130795
common_ancestor       0.975556  0.975556  ...  0.107985  0.107985
family_ancestor       0.971111  0.968889  ...  0.100641  0.123913
reference0            0.973333  0.968889  ...  0.112408  0.128697
reference1            0.970000  0.964444  ...  0.099852  0.122707
reference2            0.969444  0.964444  ...  0.115363  0.130037
test_donor            0.975556  0.973333  ...  0.105404  0.140104

[7 rows x 6 columns]